# Notebook 11 (Revised) — Leakage-Safe Neural Forecasting

## Milestone 1: feature parity, 13×9 sequences and fold-specific scaling

This is the **new revised neural development track**, distinct from `11_deep_learning_models.ipynb`, which remains the historical executed experiment. Development uses the frozen source protocol from `09_modelling_dataset_and_protocol.ipynb` and revised Notebook 10 at commit `0215291`.

**Scientific target:** predict SOLETE PV output `P_Solar[kW]` at **15, 30, 60 minutes** with nested F0–F4 predictor families. Primary representation comparison: **F3 → F4**.

**Frozen availability-safe development selection:** 3 expanding-window folds from `src.forecasting_protocol.make_rolling_origin_masks`, 60-minute maximum-horizon label-availability guard, **unweighted** mean of the three fold MAEs for candidate ranking. Only fold train/validation labels may be used. **Locked terminal holdout: June–August 2019 (26,496 rows); excluded from all folds.**

**Neural input parity:** 13×9 issue-time observed history ordered from `t−60` to `t`; auxiliary widths 1/16/21/30/127 for F0–F4. The first auxiliary is PV at `t−24h`, while additional deterministic features refer to forecast-valid time.

**Milestone 1 does not train, predict, select hyperparameters, refit, write scientific results, or evaluate the terminal holdout.** It reconstructs the same canonical predictor tables and demonstrates fold-specific, training-only scalers with no test rows in their inputs. Milestone 2 will freeze architecture grids and conduct resource pilots; Milestone 3 will undertake full neural development CV. Historical validation-selected architectures and epochs are **not** imported as winners.

Run with the pinned project's `.venv` from the repository root or `notebooks/`. Required local inputs: SOLETE 5-minute HDF5, `ephemeris/de440s.bsp`, unchanged Notebook 09, and `src/revised_neural_cv.py` (provided separately with this notebook).

## A. Canonical Notebook 09 source-integrity preflight

The following preflight and canonical dataset-building cells are **source-identical** to the executed revised Notebook 10. Notebook 09 source hashes fail closed if historical construction is modified. Reusing these cells preserves feature parity without introducing a second, slightly different astronomy implementation.

In [1]:
from pathlib import Path
import hashlib
import json

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
REFERENCE_NOTEBOOK = PROJECT_ROOT / "notebooks" / "09_modelling_dataset_and_protocol.ipynb"

# Canonical construction: the first FIVE code cells in Notebook 09 at
# validated project commit 369a9d9.  Do not permit silent code drift.
NOTEBOOK09_CODE_SHA256 = (
    "18d232021455524939be8e14fc67842819ea4b8701a41a279b8997a47768bbee",
    "6add41c3649cfd7300a1fa04e0a966f30f7fc78c2a4b4bbaa054056358ddc21b",
    "724b7cdf9cd01790c9a0c9371967ff1c264c0f0d9730d1ef8d9be30dd359456d",
    "303947573618f2bcc5c280e2dd6e2d392805dc194073d51a8e4c84c7b4f3a373",
    "58ab56827f81c210df0ec662208f217baa3043d22c1940b028b041407efa9f09",
)

if not REFERENCE_NOTEBOOK.is_file():
    raise FileNotFoundError(f"Cannot find canonical Notebook 09: {REFERENCE_NOTEBOOK}")
with REFERENCE_NOTEBOOK.open(encoding="utf-8") as handle:
    _reference = json.load(handle)
reference_code_cells = [
    "".join(cell.get("source", []))
    for cell in _reference["cells"]
    if cell["cell_type"] == "code"
]
actual_hashes = tuple(
    hashlib.sha256(source.encode("utf-8")).hexdigest()
    for source in reference_code_cells[:5]
)
if actual_hashes != NOTEBOOK09_CODE_SHA256:
    raise RuntimeError(
        "Notebook 09 canonical construction source changed since commit 369a9d9. "
        "Stop: review and resynchronize the revised Notebook 10 instead of "
        "training with mismatched feature definitions."
    )
print("Canonical Notebook 09 source hash parity: PASSED (5 construction cells)")
print("Notebook 09 source: read-only")


Canonical Notebook 09 source hash parity: PASSED (5 construction cells)
Notebook 09 source: read-only


In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import swisseph as swe
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete
from src.preprocessing import (
    build_cleaned_solete_features,
    build_gregorian_features,
)
import src.astronomy_features as astronomy_features
import src.vedic_calendar as vedic_calendar
from src.astronomy_features import (
    build_lunar_f3_features,
    build_solar_f2_features,
    load_jpl_ephemeris,
    solete_astronomy_reference_times,
    tropical_longitudes_for_solete,
)
from src.vedic_calendar import (
    build_panchang_f4_raw,
    lahiri_ayanamsha_for_utc_times,
)
from src.feature_families import (
    encode_panchang_fixed_vocabulary,
    PANCHANG_INDICATOR_COUNT,
)

DATA_5MIN = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_5min.h5"
EPHEMERIS_FILE = PROJECT_ROOT / "ephemeris" / "de440s.bsp"

for path in [DATA_5MIN, EPHEMERIS_FILE]:
    if not path.exists():
        raise FileNotFoundError(path)

# ------------------------------------------------------------------
# Source-parity guard.
#
# Notebook 07 froze pvlib altitude=0 m and delta_t=None.
# Notebook 08 froze standard Lahiri with FLG_NONUT.
#
# Stop immediately if reusable source files still implement the older
# conventions. A source-sync helper is supplied alongside this notebook.
# ------------------------------------------------------------------

if not hasattr(
    astronomy_features,
    "SOLETE_SOLAR_ALTITUDE_CONVENTION_M",
):
    raise RuntimeError(
        "src/astronomy_features.py is older than revised Notebook 07. "
        "Apply the supplied source-sync helper before running Notebook 09."
    )

assert astronomy_features.SOLETE_SOLAR_ALTITUDE_CONVENTION_M == 0.0

assert (
    vedic_calendar.LAHIRI_AYANAMSHA_FLAGS
    & swe.FLG_NONUT
), (
    "src/vedic_calendar.py still uses the older nutation-included "
    "Lahiri convention. Apply the supplied source-sync helper first."
)

df_raw = load_solete(DATA_5MIN).sort_index()

assert df_raw.index.is_monotonic_increasing
assert not df_raw.index.has_duplicates
assert df_raw.index.tz is not None

expected_step = pd.Timedelta(minutes=5)
observed_steps = df_raw.index.to_series().diff().dropna()
assert (observed_steps == expected_step).all(), (
    "SOLETE modelling timeline is not a complete 5-minute grid."
)

print("SOLETE:", df_raw.shape)
print("Period:", df_raw.index.min(), "to", df_raw.index.max())
print("Astronomy offset:", astronomy_features.SOLETE_ASTRONOMY_OFFSET)
print("Solar altitude convention [m]:",
      astronomy_features.SOLETE_SOLAR_ALTITUDE_CONVENTION_M)
print("Lahiri flags:", vedic_calendar.LAHIRI_AYANAMSHA_FLAGS)


SOLETE: (131617, 11)
Period: 2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00
Astronomy offset: 0 days 00:02:00
Solar altitude convention [m]: 0.0
Lahiri flags: 66


In [3]:
RESOLUTION_MINUTES = 5
FORECAST_HORIZONS = {
    15: 3,
    30: 6,
    60: 12,
}

PRIMARY_TARGET = "P_Solar[kW]"

F0_HISTORY_SOURCES = {
    "P_Solar_kw": "P_Solar[kW]",
    "GHI_kw1m2": "GHI[kW1m2]",
    "POA_Irr_kw1m2": "POA Irr[kW1m2]",
    "temperature_degC": "TEMPERATURE[degC]",
    "humidity_clean": "Humidity_clean",
    "pressure_clean_mbar": "Pressure_clean[mbar]",
    "wind_speed_m1s": "WIND_SPEED[m1s]",
    "wind_direction_sin": "wind_direction_sin",
    "wind_direction_cos": "wind_direction_cos",
}

RECENT_LAGS = tuple(range(13))
DAILY_LAG_STEPS = 24 * 60 // RESOLUTION_MINUTES

assert DAILY_LAG_STEPS == 288

df_clean = build_cleaned_solete_features(df_raw)

required_f0_sources = list(F0_HISTORY_SOURCES.values())
assert not df_clean[required_f0_sources].isna().any().any()

# F0: nine channels at t,t-5,...,t-60 plus P_Solar(t-24h).
f0_columns = {}

for feature_name, source_column in F0_HISTORY_SOURCES.items():
    for lag in RECENT_LAGS:
        f0_columns[f"{feature_name}_lag_{lag}"] = (
            df_clean[source_column].shift(lag)
        )

f0_columns["P_Solar_kw_issue_minus_24h"] = (
    df_clean[PRIMARY_TARGET].shift(DAILY_LAG_STEPS)
)

df_f0_full = pd.DataFrame(f0_columns, index=df_clean.index)

assert df_f0_full.shape[1] == 118

first_complete_f0_issue = df_f0_full.dropna().index.min()
expected_first_complete_f0_issue = pd.Timestamp(
    "2018-06-02 00:00:00",
    tz="UTC",
)
assert first_complete_f0_issue == expected_first_complete_f0_issue

print("F0 full-timeline shape:", df_f0_full.shape)
print("First row with complete F0:", first_complete_f0_issue)


F0 full-timeline shape: (131617, 118)
First row with complete F0: 2018-06-02 00:00:00+00:00


In [4]:
STUDY_START = pd.Timestamp("2018-06-01 00:00:00", tz="UTC")
LOCKED_TEST_START = pd.Timestamp("2019-06-01 00:00:00", tz="UTC")
STUDY_END = pd.Timestamp("2019-09-01 00:00:00", tz="UTC")

base_modelling_tables = {}

for horizon_minutes, horizon_steps in FORECAST_HORIZONS.items():
    table = df_f0_full.copy()

    table["issue_time"] = table.index
    table["valid_time"] = (
        table["issue_time"]
        + pd.Timedelta(minutes=horizon_minutes)
    )
    table["target_P_Solar_kW"] = (
        df_clean[PRIMARY_TARGET].shift(-horizon_steps)
    )
    table["horizon_minutes"] = horizon_minutes

    usable = (
        table[df_f0_full.columns].notna().all(axis=1)
        & table["target_P_Solar_kW"].notna()
        & table["valid_time"].ge(STUDY_START)
        & table["valid_time"].lt(STUDY_END)
    )

    table = table.loc[usable].copy()

    table["final_partition"] = np.where(
        table["valid_time"] < LOCKED_TEST_START,
        "development",
        "locked_test",
    )

    # Direct target-alignment verification.
    sample_positions = np.linspace(
        0, len(table) - 1, 5, dtype=int
    )
    for position in sample_positions:
        valid_time = table["valid_time"].iloc[position]
        observed = table["target_P_Solar_kW"].iloc[position]
        expected = df_clean.loc[valid_time, PRIMARY_TARGET]
        assert np.isclose(observed, expected, atol=0.0, rtol=0.0)

    base_modelling_tables[horizon_minutes] = table

population_summary = []

for horizon, table in base_modelling_tables.items():
    population_summary.append({
        "horizon_minutes": horizon,
        "rows": len(table),
        "development_rows":
            int(table["final_partition"].eq("development").sum()),
        "locked_test_rows":
            int(table["final_partition"].eq("locked_test").sum()),
        "first_issue": table["issue_time"].min(),
        "first_valid": table["valid_time"].min(),
        "last_valid": table["valid_time"].max(),
    })

display(pd.DataFrame(population_summary))


,horizon_minutes,rows,development_rows,locked_test_rows,first_issue,first_valid,last_valid
0,15,131325,104829,26496,2018-06-02 00:00:00+00:00,2018-06-02 00:15:00+00:00,2019-08-31 23:55:00+00:00
1,30,131322,104826,26496,2018-06-02 00:00:00+00:00,2018-06-02 00:30:00+00:00,2019-08-31 23:55:00+00:00
2,60,131316,104820,26496,2018-06-02 00:00:00+00:00,2018-06-02 01:00:00+00:00,2019-08-31 23:55:00+00:00


In [5]:
ephemeris = load_jpl_ephemeris(EPHEMERIS_FILE)

# F1: no astronomical offset.
df_f1_increment_full = build_gregorian_features(df_clean.index)
assert df_f1_increment_full.shape == (len(df_clean), 15)

# F2/F3: module applies row timestamp + 2 min.
df_f2_increment_full = build_solar_f2_features(df_clean.index)
df_f3_increment_full = build_lunar_f3_features(
    df_clean.index,
    ephemeris,
)

assert df_f2_increment_full.shape == (len(df_clean), 5)
assert df_f3_increment_full.shape == (len(df_clean), 9)

# F4 parents at the same +2 min astronomical reference.
df_tropical_full = tropical_longitudes_for_solete(
    df_clean.index,
    ephemeris,
)

astro_times = solete_astronomy_reference_times(df_clean.index)

ayanamsha = lahiri_ayanamsha_for_utc_times(astro_times)

df_ayanamsha = pd.Series(
    ayanamsha,
    index=df_clean.index,
    name="lahiri_ayanamsha_nonut_deg",
)

# Canonical Notebook 08 first-point guard.
KNOWN_FIRST_LAHIRI_NONUT = 24.11431760121269

assert np.isclose(
    df_ayanamsha.iloc[0],
    KNOWN_FIRST_LAHIRI_NONUT,
    atol=1e-10,
    rtol=0.0,
)

df_f4_raw_full = build_panchang_f4_raw(
    sun_tropical_longitude_deg=
        df_tropical_full["sun_tropical_longitude_deg"],
    moon_tropical_longitude_deg=
        df_tropical_full["moon_tropical_longitude_deg"],
    ayanamsha_deg=df_ayanamsha,
)

assert df_f4_raw_full.shape == (len(df_clean), 5)
assert not df_f4_raw_full.isna().any().any()

first_f4 = df_f4_raw_full.iloc[0]
assert int(first_f4["tithi"]) == 18
assert first_f4["paksha"] == "Krishna"
assert first_f4["karana"] == "Vanija"
assert first_f4["nakshatra"] == "Mula"
assert first_f4["yoga"] == "Shubha"

print("F1 increment:", df_f1_increment_full.shape)
print("F2 increment:", df_f2_increment_full.shape)
print("F3 increment:", df_f3_increment_full.shape)
print("Raw F4:", df_f4_raw_full.shape)
print("First Lahiri NONUT:", df_ayanamsha.iloc[0])
print("Notebook 07/08 source parity: PASSED")


F1 increment: (131617, 15)
F2 increment: (131617, 5)
F3 increment: (131617, 9)
Raw F4: (131617, 5)
First Lahiri NONUT: 24.11431760121269
Notebook 07/08 source parity: PASSED


In [6]:
EXPECTED_FEATURE_COUNTS = {
    "F0": 118,
    "F1": 133,
    "F2": 138,
    "F3": 147,
    "F4": 244,
}

assert PANCHANG_INDICATOR_COUNT == 97

feature_family_tables = {}
panchang_onehot_tables = {}

for horizon, base in base_modelling_tables.items():
    issue_index = base.index
    valid_index = pd.DatetimeIndex(
        base["valid_time"],
        name="valid_time",
    )

    f0 = df_f0_full.loc[issue_index].copy()

    f1_inc = df_f1_increment_full.loc[valid_index].copy()
    f1_inc.index = issue_index
    f1_inc = f1_inc.add_prefix("valid_")

    f2_inc = df_f2_increment_full.loc[valid_index].copy()
    f2_inc.index = issue_index
    f2_inc = f2_inc.add_prefix("valid_")

    f3_inc = df_f3_increment_full.loc[valid_index].copy()
    f3_inc.index = issue_index
    f3_inc = f3_inc.add_prefix("valid_")

    f4_raw = df_f4_raw_full.loc[valid_index].copy()
    f4_raw.index = issue_index
    f4_raw = f4_raw.add_prefix("valid_")

    f4_onehot = encode_panchang_fixed_vocabulary(f4_raw)

    assert f4_onehot.shape[1] == 97
    assert (f4_onehot.sum(axis=1) == 5).all()

    F0 = f0
    F1 = pd.concat([F0, f1_inc], axis=1)
    F2 = pd.concat([F1, f2_inc], axis=1)
    F3 = pd.concat([F2, f3_inc], axis=1)
    F4 = pd.concat([F3, f4_onehot], axis=1)

    families = {
        "F0": F0,
        "F1": F1,
        "F2": F2,
        "F3": F3,
        "F4": F4,
    }

    for family, table in families.items():
        assert table.index.equals(issue_index)
        assert table.shape[1] == EXPECTED_FEATURE_COUNTS[family]
        assert not table.isna().any().any()

    assert list(F1.columns[:118]) == list(F0.columns)
    assert list(F2.columns[:133]) == list(F1.columns)
    assert list(F3.columns[:138]) == list(F2.columns)
    assert list(F4.columns[:147]) == list(F3.columns)

    feature_family_tables[horizon] = families
    panchang_onehot_tables[horizon] = f4_onehot

print("Nested feature construction: PASSED")

for horizon, families in feature_family_tables.items():
    print(
        horizon,
        {name: table.shape for name, table in families.items()},
    )


Nested feature construction: PASSED
15 {'F0': (131325, 118), 'F1': (131325, 133), 'F2': (131325, 138), 'F3': (131325, 147), 'F4': (131325, 244)}
30 {'F0': (131322, 118), 'F1': (131322, 133), 'F2': (131322, 138), 'F3': (131322, 147), 'F4': (131322, 244)}
60 {'F0': (131316, 118), 'F1': (131316, 133), 'F2': (131316, 138), 'F3': (131316, 147), 'F4': (131316, 244)}


## B. Cross-check canonical features and time gaps

Use the independent Notebook 10 validations for exact F0 history and future targets, sampled deterministic feature parity, nested F0–F4 columns, 3-fold chronological boundaries, label-availability gaps, and the 26,496-row untouched holdout. The code is unchanged from the executed revised Notebook 10.

In [7]:
# Cross-check Notebook 09's independently written F0 and future-target
# calculations against the tested reusable forecasting protocol.
from src.forecasting_protocol import (
    F0_FEATURE_COLUMNS,
    FORECAST_HORIZONS as SOURCE_FORECAST_HORIZONS,
    build_base_modelling_tables as source_build_base,
    build_f0_history as source_build_f0,
    build_forecast_targets as source_build_targets,
)

assert FORECAST_HORIZONS == SOURCE_FORECAST_HORIZONS
source_f0 = source_build_f0(df_clean)
assert tuple(source_f0.columns) == F0_FEATURE_COLUMNS

# One known schema alias: both expressions mean P_Solar(issue_time - 24h).
# Do not silently relabel any other features.
source_f0 = source_f0.rename(
    columns={"P_Solar_kw_lag_288": "P_Solar_kw_issue_minus_24h"}
)
pd.testing.assert_frame_equal(source_f0, df_f0_full, check_exact=True)

source_targets = source_build_targets(df_clean)
source_base_tables = source_build_base(df_clean)
for horizon, base in base_modelling_tables.items():
    other = source_base_tables[horizon]
    assert base.index.equals(other.index), f"{horizon}: sample index mismatch"
    # Ignore only the two mutually incompatible *legacy* split metadata columns;
    # revised models use availability-safe CV masks below.
    compare_columns = list(df_f0_full.columns) + [
        "issue_time", "valid_time", "horizon_minutes", "target_P_Solar_kW"
    ]
    other_comparable = other.rename(
        columns={"P_Solar_kw_lag_288": "P_Solar_kw_issue_minus_24h"}
    )[compare_columns]
    pd.testing.assert_frame_equal(
        base[compare_columns], other_comparable, check_exact=True
    )
    pd.testing.assert_series_equal(
        base["target_P_Solar_kW"],
        source_targets[horizon].loc[base.index, "target_P_Solar_kW"],
        check_exact=True,
    )
print("F0 (118 columns) exact numerical/source parity: PASSED")
print("All horizon issue times, valid times and targets: PASSED")


F0 (118 columns) exact numerical/source parity: PASSED
All horizon issue times, valid times and targets: PASSED


In [8]:
# Independently RECOMPUTE deterministic increments at boundary-focused
# probe times. This detects wrong issue/valid alignment or category encoding
# without recomputing the full ephemeris three additional times.

def _compare_increment(actual, expected, *, discrete=False):
    pd.testing.assert_index_equal(actual.columns, expected.columns)
    assert actual.shape == expected.shape
    assert not actual.isna().any().any()
    assert not expected.isna().any().any()
    if discrete:
        pd.testing.assert_frame_equal(
            actual.reset_index(drop=True), expected.reset_index(drop=True),
            check_exact=True,
        )
    else:
        np.testing.assert_allclose(
            actual.to_numpy(dtype=np.float64),
            expected.to_numpy(dtype=np.float64),
            atol=1e-8, rtol=1e-10,
        )


_PROBE_BOUNDARIES = pd.DatetimeIndex([
    "2018-12-01 00:00:00+00:00",
    "2019-02-01 00:00:00+00:00",
    "2019-04-01 00:00:00+00:00",
    "2019-06-01 00:00:00+00:00",
    "2019-09-01 00:00:00+00:00",
])

probe_summary = []
for horizon, base in base_modelling_tables.items():
    all_valid = pd.DatetimeIndex(base["valid_time"])
    indices = {0, len(base) - 1}
    indices.update(np.linspace(0, len(base) - 1, num=17, dtype=int))
    for boundary in _PROBE_BOUNDARIES:
        position = int(all_valid.searchsorted(boundary))
        indices.update(p for p in (position - 1, position, position + 1)
                       if 0 <= p < len(base))
    positions = np.array(sorted(indices), dtype=int)
    issues = base.index[positions]
    valid = pd.DatetimeIndex(base["valid_time"].iloc[positions])
    families = feature_family_tables[horizon]

    greg = build_gregorian_features(valid).add_prefix("valid_")
    solar = build_solar_f2_features(valid).add_prefix("valid_")
    lunar = build_lunar_f3_features(valid, ephemeris).add_prefix("valid_")

    _compare_increment(
        families["F1"].loc[issues, greg.columns], greg
    )
    _compare_increment(
        families["F2"].loc[issues, solar.columns], solar
    )
    _compare_increment(
        families["F3"].loc[issues, lunar.columns], lunar
    )

    tropical = tropical_longitudes_for_solete(valid, ephemeris)
    astronomy_times = solete_astronomy_reference_times(valid)
    ayan = pd.Series(
        lahiri_ayanamsha_for_utc_times(astronomy_times),
        index=valid,
        name="lahiri_ayanamsha_nonut_deg",
    )
    panchang_raw = build_panchang_f4_raw(
        sun_tropical_longitude_deg=tropical["sun_tropical_longitude_deg"],
        moon_tropical_longitude_deg=tropical["moon_tropical_longitude_deg"],
        ayanamsha_deg=ayan,
    ).add_prefix("valid_")
    encoded = encode_panchang_fixed_vocabulary(panchang_raw)
    _compare_increment(
        families["F4"].loc[issues, encoded.columns], encoded,
        discrete=True,
    )
    assert (encoded.sum(axis=1) == 5).all()
    probe_summary.append({
        "horizon_minutes": horizon,
        "probe_rows": len(positions),
        "F1-F3_numerical_recompute": "PASS",
        "F4_exact_onehot_recompute": "PASS",
    })

display(pd.DataFrame(probe_summary))
print("Sampled F1-F4 re-computation and row alignment: PASSED")


,horizon_minutes,probe_rows,F1-F3_numerical_recompute,F4_exact_onehot_recompute
0,15,29,PASS,PASS
1,30,29,PASS,PASS
2,60,29,PASS,PASS


Sampled F1-F4 re-computation and row alignment: PASSED


In [9]:
# Build masks solely from the finalized source protocol. The legacy
# base['split'] or notebook 10 split_labels must never control training.
from src.forecasting_protocol import (
    CV_FOLDS,
    EXPECTED_CV_COUNTS_BY_HORIZON,
    EXPECTED_TERMINAL_REFIT_COUNTS_BY_HORIZON,
    MODEL_AVAILABILITY_GAP,
    make_rolling_origin_masks,
    make_terminal_refit_mask,
    terminal_holdout_mask,
    validate_rolling_origin_counts,
)

assert MODEL_AVAILABILITY_GAP == pd.Timedelta(minutes=60)
assert len(CV_FOLDS) == 3

EXPECTED_TOTAL = {15: 131325, 30: 131322, 60: 131316}
EXPECTED_DEVELOPMENT = {15: 104829, 30: 104826, 60: 104820}
cv_masks = {}
refit_masks = {}
holdout_masks = {}
summary = []
for horizon, base in base_modelling_tables.items():
    assert len(base) == EXPECTED_TOTAL[horizon]
    valid = base["valid_time"]
    issue = base["issue_time"]
    holdout = terminal_holdout_mask(valid)
    refit = make_terminal_refit_mask(valid)
    folds = make_rolling_origin_masks(valid)

    assert int(holdout.sum()) == 26496
    assert int((~holdout).sum()) == EXPECTED_DEVELOPMENT[horizon]
    assert int(refit.sum()) == EXPECTED_TERMINAL_REFIT_COUNTS_BY_HORIZON[horizon]
    assert valid.loc[refit].max() < issue.loc[holdout].min()
    assert validate_rolling_origin_counts(valid, horizon) == (
        EXPECTED_CV_COUNTS_BY_HORIZON[horizon]
    )
    assert len(families := feature_family_tables[horizon]) == 5
    assert all(frame.index.equals(base.index) for frame in families.values())
    assert (base.index.to_series().diff().dropna() == pd.Timedelta(minutes=5)).all()

    cv_masks[horizon] = {}
    for spec, fold in zip(CV_FOLDS, folds, strict=True):
        train = fold["train_mask"]
        val = fold["validation_mask"]
        assert not (train & val).any()
        assert not ((train | val) & holdout).any()
        assert valid.loc[train].max() < issue.loc[val].min()
        assert valid.loc[val].max() < issue.loc[holdout].min()
        cv_masks[horizon][spec["fold"]] = {"train": train, "validation": val}
        summary.append({
            "horizon": horizon,
            "fold": spec["fold"],
            "train": int(train.sum()),
            "validation": int(val.sum()),
            "holdout": int(holdout.sum()),
        })
    refit_masks[horizon] = refit
    holdout_masks[horizon] = holdout

display(pd.DataFrame(summary))
print("Three-fold availability-safe protocol: PASSED")
print("Original observed timeline and all 26,496 terminal rows: PRESERVED")
print("Milestone 1 passed. No model has been fitted or scored.")


,horizon,fold,train,validation,holdout
0,15,1,52401,17856,26496
1,15,2,70257,16992,26496
2,15,3,87249,17556,26496
3,30,1,52398,17856,26496
4,30,2,70254,16992,26496
5,30,3,87246,17556,26496
6,60,1,52392,17856,26496
7,60,2,70248,16992,26496
8,60,3,87240,17556,26496


Three-fold availability-safe protocol: PASSED
Original observed timeline and all 26,496 terminal rows: PRESERVED
Milestone 1 passed. No model has been fitted or scored.


## C. Neural input contract and development-only fold materialization

Repackage the *same* flat F0–F4 predictors into the original historical two-branch design. Each 13-step historical sequence is ordered chronologically oldest-to-newest. No observed measurements from times after the forecast issue are permitted. The daily PV lag remains an auxiliary scalar.

Only rows belonging to the requested **training + validation fold** are materialized. The terminal targets are replaced with missing values **before** invoking the neural input helper. Distinct folds will be scaled independently; future model fitting will use only these prepared tensors. The smoke audit covers the smallest and largest families, not model training.

In [10]:
import numpy as np
import pandas as pd
from src.revised_neural_cv import (
    AUXILIARY_DIMENSIONS, FAMILY_DIMENSIONS, F0_COLUMNS,
    prepare_development_fold,
)

assert tuple(feature_family_tables[15]["F0"].columns) == F0_COLUMNS
assert dict(EXPECTED_FEATURE_COUNTS) == FAMILY_DIMENSIONS

# Source fold masks (NOT historical split labels) and terminal-target masking.
neural_fold_smokes = {}
for horizon, family, fold_number in ((15, "F0", 1), (60, "F4", 3)):
    base = base_modelling_tables[horizon]
    forbidden = holdout_masks[horizon]
    assert int(forbidden.sum()) == 26_496
    # The helper receives no terminal labels, not even for scaler preparation.
    masked_target = base["target_P_Solar_kW"].mask(forbidden)
    assert masked_target.loc[forbidden].isna().all()
    fold = make_rolling_origin_masks(base["valid_time"])[fold_number - 1]
    payload = prepare_development_fold(
        X=feature_family_tables[horizon][family],
        masked_target=masked_target,
        issue_time=base["issue_time"],
        valid_time=base["valid_time"],
        forbidden_mask=forbidden,
        fold=fold,
        horizon_minutes=horizon,
        family=family,
    )
    assert (payload["train_rows"], payload["validation_rows"]) == (
        EXPECTED_CV_COUNTS_BY_HORIZON[horizon][fold_number - 1]
    )
    assert payload["sequence"].shape == (
        payload["train_rows"] + payload["validation_rows"], 13, 9
    )
    assert payload["auxiliary"].shape[1] == AUXILIARY_DIMENSIONS[family]
    assert payload["target_kw"].shape[0] == len(payload["sequence"])
    assert len(payload["issue_time"]) == len(payload["sequence"])
    assert len(payload["binary_positions"]) == (0 if family == "F0" else 99)
    neural_fold_smokes[(horizon, family, fold_number)] = payload
    print(f"{horizon:2d} min {family} fold {fold_number}: "
          f"train={payload['train_rows']}, validation={payload['validation_rows']}, "
          f"sequence={payload['sequence'].shape}, auxiliary={payload['auxiliary'].shape}")

print("Neural input parity / forbidden-holdout isolation: PASSED")

15 min F0 fold 1: train=52401, validation=17856, sequence=(70257, 13, 9), auxiliary=(70257, 1)
60 min F4 fold 3: train=87240, validation=17556, sequence=(104796, 13, 9), auxiliary=(104796, 127)
Neural input parity / forbidden-holdout isolation: PASSED


## D. Fold-specific training-only scaling smoke audit (no fitting)

Reuse only the **scaling functions** from `src/neural_models.py`, not its historical selected model architectures or epoch maps. This step fits per-channel/continuous-feature means and standard deviations from the current fold's **training rows only**. Preserve 0/1 indicators unchanged: `valid_utc_is_weekend`, `valid_solar_daylight`, and 97 fixed Panchang one-hot columns. The target is transformed only to prepare for future development training; no network is trained or scored here. All reporting later must inverse-transform predictions to kW.

The historical scaling rule uses population standard deviation (`ddof=0`) and fails closed for continuous zero-variance columns.

In [11]:
from src.neural_models import (
    fit_train_only_scalers,
    apply_train_only_scalers,
    inverse_target_scaling,
)

for case, payload in neural_fold_smokes.items():
    stats = fit_train_only_scalers(
        payload["sequence"],
        payload["auxiliary"],
        payload["target_kw"],
        payload["train_mask"],
        continuous_positions=payload["continuous_positions"],
        binary_positions=payload["binary_positions"],
    )
    scaled = apply_train_only_scalers(
        payload["sequence"], payload["auxiliary"],
        payload["target_kw"], stats,
    )
    assert stats["n_train"] == payload["train_rows"]
    assert scaled["sequence"].shape == payload["sequence"].shape
    assert scaled["auxiliary"].shape == payload["auxiliary"].shape
    np.testing.assert_allclose(
        stats["sequence_mean"],
        payload["sequence"][payload["train_mask"]].mean(axis=(0, 1), dtype=np.float64),
        atol=1e-10, rtol=0,
    )
    pos = payload["binary_positions"]
    if pos.size:
        np.testing.assert_array_equal(
            scaled["auxiliary"][:, pos], payload["auxiliary"][:, pos]
        )
    restored = inverse_target_scaling(scaled["target_scaled"], stats)
    np.testing.assert_allclose(restored, payload["target_kw"], atol=2e-5, rtol=1e-5)
    print(f"{case}: train-only scaling and kW round-trip PASSED")

print("NEURAL MILESTONE 1: PASSED — no model fits, predictions, terminal evaluation or writes")

(15, 'F0', 1): train-only scaling and kW round-trip PASSED
(60, 'F4', 3): train-only scaling and kW round-trip PASSED
NEURAL MILESTONE 1: PASSED — no model fits, predictions, terminal evaluation or writes


## Milestone 1 completion gate

**Stop here.** Do not append or run historical Notebook 11 model-selection/test cells. Save this notebook with its executed outputs and report the printed `NEURAL MILESTONE 1: PASSED` message. Next we will freeze a resource-conscious LSTM/TCN candidate grid, seed/epoch policy, and resumption strategy **before** any neural validation model selection.

No historical terminal test results or previously selected epoch maps may influence the revised search. The historical experiment remains available solely for methodology comparison and provenance. The next milestone must ensure each fold's feature/target scalers are independently train-fitted and that all candidate ranking uses mean **kW MAE**, not standardized-target MAE.

## Milestone 2 — Frozen neural-CV design and two one-epoch resource pilots

**Scientific scope:** This is **not** architecture selection. All original Milestone 1 cells and outputs above are preserved. The frozen candidate shapes are declared in `src/revised_neural_training.py`; no historical *winning* architecture or frozen epoch-map is reused as a revised winner.

Full revised nested-CV will select the epoch count inside each **outer training** window, using an inner chronological tail (10%) with a separate **60-minute label-availability gap** before the first inner early-stopping validation time. The model would then be **reinitialized** and fitted on **all of that outer training fold** for its selected number of epochs, with scalers re-fitted on the outer training fold only. Outer validation labels would be used **once**, for unbiased candidate evaluation in physical kW. Candidate comparison requires three complete outer folds and uses the unweighted mean of outer-fold kW MAEs, with configuration order breaking exact ties.

**This milestone only:** check the inner masks and run two one-epoch, bounded **LSTM_C1 / TCN_C1** resource pilots using **15-minute F0, outer fold 1**, already materialized in Milestone 1. Each pilot uses up to **2,048 inner-training** and **512 inner-early-stopping** rows; neither pilot uses the outer-validation population. The 1-epoch diagnostic is **not** a CV score and **must not** guide model selection. Checkpoints are written to `results/development_only/revised_neural_pilot_v1/` with strict data/source identities; repeats skip matching completed pilots, while mismatches fail closed. No terminal test/refit is performed.

In [12]:
# Freeze revised selection protocol before any neural model fit.
from src.revised_neural_training import (
    LSTM_CANDIDATES, TCN_CANDIDATES,
    INNER_VALIDATION_FRACTION, INNER_LABEL_GAP_MINUTES,
    MAX_SELECTION_EPOCHS, EARLY_STOPPING_PATIENCE,
    EPOCH_RANKING, OUTER_CV_RANKING,
    PILOT_MAX_TRAIN_ROWS, PILOT_MAX_INNER_VAL_ROWS,
    make_inner_split, pilot_indices,
)

assert len(LSTM_CANDIDATES) == len(TCN_CANDIDATES) == 4
assert INNER_VALIDATION_FRACTION == 0.10
assert INNER_LABEL_GAP_MINUTES == 60
assert MAX_SELECTION_EPOCHS == 80 and EARLY_STOPPING_PATIENCE == 8

# Earlier Milestone 1 has already assembled this development-only payload.
resource_payload = neural_fold_smokes[(15, "F0", 1)]
inner = make_inner_split(resource_payload)
fit_rows, inner_val_rows = pilot_indices(resource_payload)
assert len(fit_rows) <= PILOT_MAX_TRAIN_ROWS
assert len(inner_val_rows) <= PILOT_MAX_INNER_VAL_ROWS
assert inner["inner_train_mask"].sum() < resource_payload["train_rows"]
assert not (inner["inner_train_mask"] & resource_payload["validation_mask"]).any()
assert not (inner["inner_validation_mask"] & resource_payload["validation_mask"]).any()
print("Inner chronology and outer-validation exclusion: PASSED")
print("Outer train:", resource_payload["train_rows"],
      "Inner-fit:", int(inner["inner_train_mask"].sum()),
      "Embargo:", inner["embargo_rows"],
      "Inner early-stop:", int(inner["inner_validation_mask"].sum()))
print("Pilot caps:", len(fit_rows), "train +", len(inner_val_rows), "inner validation")
print("Future complete-CV ranking:", OUTER_CV_RANKING)
print("No model fitted in this cell.")

Inner chronology and outer-validation exclusion: PASSED
Outer train: 52401 Inner-fit: 47148 Embargo: 12 Inner early-stop: 5241
Pilot caps: 2048 train + 512 inner validation
Future complete-CV ranking: unweighted mean of 3 outer-fold kW MAEs; then config index
No model fitted in this cell.


### Resource pilot — at most two bounded one-epoch fits

Run the next cell **once** after the earlier preflight and mask checks have passed. It may take a few minutes on a CPU. It runs only **LSTM_C1** and **TCN_C1**, each for one epoch, and only on a capped sample from the INNER development split. It does **not** produce any outer-fold CV MAE, select an architecture, touch the terminal holdout, or overwrite the frozen historical neural notebook.

If interrupted, rerunning this last cell skips matching completed checkpoints. If code, data, TensorFlow version, or candidate identity changes, checkpoint reuse is rejected rather than silently mixing runs. **Do not click Run All again after successful execution**—use this cell individually for resume.

In [13]:
# Exactly two one-epoch, inner-only development RESOURCE pilots (not CV).
from pathlib import Path
from src.revised_neural_training import source_fingerprint, run_resource_pilot

pilot_root = Path(PROJECT_ROOT)
pilot_outdir = (
    pilot_root / "results" / "development_only" / "revised_neural_pilot_v1"
)
source_sha = source_fingerprint(pilot_root)

pilot_results = {}
for model_name in ("LSTM", "TCN"):
    record, fitted_now = run_resource_pilot(
        payload=resource_payload,
        root=pilot_root,
        model_name=model_name,
        source_sha256=source_sha,
        output_dir=pilot_outdir,
    )
    assert record["epochs"] == 1
    assert record["train_rows"] <= 2048
    assert record["inner_validation_rows"] <= 512
    assert record["trainable_parameters"] > 0
    assert record["elapsed_seconds"] > 0
    pilot_results[model_name] = record
    print(f"{model_name}: {record['elapsed_seconds']:.1f} seconds; "
          f"params={record['trainable_parameters']}; "
          f"{'completed new pilot' if fitted_now else 'reused matching pilot checkpoint'}")

assert set(pilot_results) == {"LSTM", "TCN"}
print("NEURAL MILESTONE 2 RESOURCE PILOT: PASSED")
print("Development-only pilot checkpoints:", pilot_outdir)
print("No outer-CV scoring, terminal evaluation or final refit performed.")

LSTM: 2.8 seconds; params=6497; reused matching pilot checkpoint
TCN: 4.9 seconds; params=17857; reused matching pilot checkpoint
NEURAL MILESTONE 2 RESOURCE PILOT: PASSED
Development-only pilot checkpoints: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/development_only/revised_neural_pilot_v1
No outer-CV scoring, terminal evaluation or final refit performed.


## Milestone 2 completion gate — stop here

Wait for `NEURAL MILESTONE 2 RESOURCE PILOT: PASSED`. Share the two pilot runtimes/parameter counts. Do not begin 80-epoch model fitting or full candidate CV without reviewing the machine resource requirements and checkpoint design.

The pilot checkpoints are **not model-selection results**, and will be stored under their own development-only path. After success, the next stage will implement and regression-test **two-stage inner-epoch selection and outer-fold scoring**, restart-safe complete CV records, and incremental execution limits. Rebuilding astronomical features on every notebook Run All is unnecessary once these data are resident in memory.

## Milestone 3 — Controlled full-size nested CV gates (disabled by default)

**Not full neural-CV selection.** This milestone enables exactly **one** full-size
15-minute/F0/fold-1 candidate (C1) per architecture, after explicit opt-in.
A 10% chronological inner validation split with a 60-minute label gap selects
an epoch from at most 80 epochs. A **fresh** model is then refitted with a
fresh scaler on all 52,401 outer-training rows for that many epochs, and
scores the 17,856 outer-validation rows once. Neither stage accesses the
June–August 2019 terminal holdout.

**Safety:** LSTM and TCN gates are **both OFF by default**. Running all cells
without changing the switches below will NOT train full-size models.
Each completed gate creates one immutable JSON checkpoint. An interrupted
training run does **not** save a partial score; the fit must then be repeated.
These single-fold results are not architecture rankings.


In [14]:
# Preflight and explicit resource-control switches. No model fitting in this cell.
from pathlib import Path
from src.revised_neural_full_cv import (
    GATE_HORIZON, GATE_FAMILY, GATE_FOLD, GATE_TRAIN_ROWS,
    GATE_VALIDATION_ROWS, GATE_KIND, gate_source_fingerprint,
    validate_fullsize_payload, run_full_size_gate,
)

M3_RUN_LSTM = False  # Set True ONLY when ready for a potentially long full-size fit.
M3_RUN_TCN = False   # Keep False until LSTM gate runtime is reviewed.

m3_root = Path(PROJECT_ROOT)
m3_outdir = m3_root / 'results/development_only/revised_neural_fullsize_gate_v1'
m3_payload = neural_fold_smokes[(15, 'F0', 1)]
m3_train_idx, m3_val_idx, m3_inner = validate_fullsize_payload(m3_payload)
assert (len(m3_train_idx), len(m3_val_idx)) == (GATE_TRAIN_ROWS, GATE_VALIDATION_ROWS)
assert (GATE_HORIZON, GATE_FAMILY, GATE_FOLD) == (15, 'F0', 1)
assert not (m3_inner['inner_validation_mask'] & m3_payload['validation_mask']).any()
assert holdout_masks[15].sum() == 26496
m3_source_sha = gate_source_fingerprint(m3_root)
print('NEURAL MILESTONE 3 PREFLIGHT: PASSED')
print('Full-size fold:', len(m3_train_idx), 'train;', len(m3_val_idx), 'outer validation')
print('Inner-fit:', int(m3_inner['inner_train_mask'].sum()),
      'inner-stop:', int(m3_inner['inner_validation_mask'].sum()),
      'embargo:', m3_inner['embargo_rows'])
print('Both full-size model gates disabled by default:', not (M3_RUN_LSTM or M3_RUN_TCN))
print('Development-only checkpoint directory:', m3_outdir)


NEURAL MILESTONE 3 PREFLIGHT: PASSED
Full-size fold: 52401 train; 17856 outer validation
Inner-fit: 47148 inner-stop: 5241 embargo: 12
Both full-size model gates disabled by default: True
Development-only checkpoint directory: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/development_only/revised_neural_fullsize_gate_v1


### Gate A — LSTM_C1, full-size 15-minute/F0/fold-1 CV

**Opt in explicitly:** change `M3_RUN_LSTM = False` to `True` in the
preflight cell, then rerun the preflight and **only this LSTM gate cell**.
Keep TCN disabled. This may take a considerable amount of time: inner epoch
selection and the fresh outer refit both train on tens of thousands of rows.
No full-grid search is enabled.


In [15]:
if M3_RUN_LSTM:
    m3_lstm_record, m3_lstm_new = run_full_size_gate(
        payload=m3_payload, root=m3_root, model_name='LSTM',
        source_sha256=m3_source_sha, output_dir=m3_outdir,
    )
    assert 1 <= m3_lstm_record['best_epoch'] <= 80
    assert m3_lstm_record['outer_training_rows'] == GATE_TRAIN_ROWS
    assert m3_lstm_record['outer_validation_rows'] == GATE_VALIDATION_ROWS
    print('LSTM FULL-SIZE GATE: PASSED', 'new fit' if m3_lstm_new else 'checkpoint reused')
    print('Best inner epoch:', m3_lstm_record['best_epoch'])
    print('Inner selection seconds:', round(m3_lstm_record['inner_selection_seconds'], 1))
    print('Outer refit seconds:', round(m3_lstm_record['outer_refit_seconds'], 1))
    print('Outer fold MAE (kW):', round(m3_lstm_record['validation_mae_kw'], 6))
else:
    print('LSTM FULL-SIZE GATE: DISABLED — no model fit performed')


LSTM FULL-SIZE GATE: DISABLED — no model fit performed


### Gate B — TCN_C1, same full-size fold

**Do not enable yet.** Once the LSTM full-size gate finishes and its runtime
and checkpoint have been checked, switch `M3_RUN_TCN = True` in the preflight
cell and run this TCN cell. Do not use the single-fold scores to select an
architecture or open the terminal holdout.


In [16]:
if M3_RUN_TCN:
    m3_tcn_record, m3_tcn_new = run_full_size_gate(
        payload=m3_payload, root=m3_root, model_name='TCN',
        source_sha256=m3_source_sha, output_dir=m3_outdir,
    )
    assert 1 <= m3_tcn_record['best_epoch'] <= 80
    assert m3_tcn_record['outer_training_rows'] == GATE_TRAIN_ROWS
    assert m3_tcn_record['outer_validation_rows'] == GATE_VALIDATION_ROWS
    print('TCN FULL-SIZE GATE: PASSED', 'new fit' if m3_tcn_new else 'checkpoint reused')
    print('Best inner epoch:', m3_tcn_record['best_epoch'])
    print('Inner selection seconds:', round(m3_tcn_record['inner_selection_seconds'], 1))
    print('Outer refit seconds:', round(m3_tcn_record['outer_refit_seconds'], 1))
    print('Outer fold MAE (kW):', round(m3_tcn_record['validation_mae_kw'], 6))
else:
    print('TCN FULL-SIZE GATE: DISABLED — no model fit performed')


TCN FULL-SIZE GATE: DISABLED — no model fit performed


## Milestone 3 checkpoint gate — stop here

When the LSTM full-size gate completes, **stop and share its best epoch,
inner-selection time, outer-refit time, and checkpoint outcome** before
running the TCN. The two full-size gates are only resource/correctness checks,
not completed model-selection grids. All 360 complete neural-CV evaluations
and the locked terminal test remain disabled.

**Do not commit the newly generated checkpoints until review**; preserve this
executed notebook's earlier outputs and never overwrite the historical
`11_deep_learning_models.ipynb`.


## Milestone 4 — Restart-safe neural development-CV campaign

This milestone enables a **single** candidate/outer-fold at a time (opt-in
only). Every completed run writes an immutable, source/data-fingerprinted
checkpoint, then subsequent calls reuse that exact record. The entire 360-fit
campaign is **disabled by default**; there is no auto-loop. Partial/interrupted
fits do not count as results. Do not use single-fold feasibility scores as
hyperparameter selections or evaluate the June–August 2019 terminal holdout.

**Provenance:** the two Milestone 3 gates are read-only audited separately;
they are NOT silently copied into the new model-selection archive because the
campaign uses a newly versioned training implementation.

**Fresh kernel:** Run the Milestone 4 preflight directly to verify imports and checkpoint count. Full Milestone 3 checkpoint integrity requires `neural_fold_smokes[(15, "F0", 1)]`; if the preflight says **PENDING**, execute the existing setup/data cells through **Section C**, then rerun the preflight. Do **not** rerun neural training.

### Safe one-click verification (recommended)

From a **fresh kernel**, select **Run All** without editing the three model-training
switches: `M3_RUN_LSTM = False`, `M3_RUN_TCN = False`, and `M4_RUN_ONE = False`.
This rebuilds the deterministic SOLETE features, reuses the two already archived
one-epoch resource pilots, verifies the saved LSTM/TCN gate checkpoints, and
prints a final run-all verification summary. It does **not** repeat the full-size
gates or launch the 360-fit neural CV campaign. The old gate training logs are
preserved in the downloaded/notebook backup; Jupyter will replace the visible
cell outputs with `DISABLED` messages when those cells are rerun. The immutable
checkpoint JSON files remain the authoritative records of the completed fits.

**Not a completed model-selection study:** The 360 evaluations remain pending
and may require many hours of computation. They must be explicitly opted into
after the checkpointed one-fit runner is verified. Do not run terminal tests.


In [17]:
# Milestone 4: standalone import bootstrap and read-only checkpoint preflight.
# Safe to run in a fresh Jupyter kernel: no fitting, no data writes.
from pathlib import Path
import sys

# VS Code/Jupyter may start inside notebooks/ instead of the repo root.
m4_root = next(
    (
        folder.resolve()
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / 'src' / 'revised_neural_campaign.py').is_file()
    ),
    None,
)
if m4_root is None:
    raise FileNotFoundError(
        'Cannot locate src/revised_neural_campaign.py. '
        'Open Notebook 11 from the SOLETE project directory.'
    )
PROJECT_ROOT = m4_root
if str(m4_root) not in sys.path:
    sys.path.insert(0, str(m4_root))

from src.revised_neural_campaign import (
    audit_milestone3_gates, campaign_source_sha256,
    validate_campaign_payload, read_campaign_progress,
    complete_rankings, run_one_cv,
)
from src.revised_neural_cv import prepare_development_fold
from src.forecasting_protocol import make_rolling_origin_masks

m4_dir = m4_root / 'results/development_only/revised_neural_cv_v1'
m4_m3_dir = m4_root / 'results/development_only/revised_neural_fullsize_gate_v1'
m4_sha = campaign_source_sha256(m4_root)
progress = read_campaign_progress(m4_dir)
m4_preflight_passed = False  # fail closed until full payload checkpoint audit

print('Milestone 4 imports: PASSED')
print('Project root:', m4_root)
print('M4 archived completed CV fits:', len(progress), 'of 360')

# A new kernel has no neural_fold_smokes until Section C has been executed.
# Read-only imports and progress inspection still work; never claim the
# source/data fingerprint audit passed unless the payload is available.
_m4_payloads = globals().get('neural_fold_smokes')
if not isinstance(_m4_payloads, dict) or (15, 'F0', 1) not in _m4_payloads:
    print('NEURAL MILESTONE 4 PREFLIGHT: PENDING — development payload not loaded.')
    print('In this kernel, execute the earlier canonical setup/data cells '
          'through Section C (Neural input contract), then rerun THIS cell.')
    print('Skip Milestone 2 pilot and Milestone 3 training cells; '
          'no neural refits are required.')
else:
    m4_m3_payload = _m4_payloads[(15, 'F0', 1)]
    m4_audit = audit_milestone3_gates(
        root=m4_root, payload=m4_m3_payload, directory=m4_m3_dir,
    )
    for name in ('LSTM', 'TCN'):
        assert m4_audit[name] is not None, f'Expected completed {name} gate'
        print(f'Milestone 3 {name}: checkpoint integrity VERIFIED; '
              f"MAE={m4_audit[name]['validation_mae_kw']:.6f} kW")
    m4_preflight_passed = True
    print('NEURAL MILESTONE 4 PREFLIGHT: PASSED — no training performed')


Milestone 4 imports: PASSED
Project root: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project
M4 archived completed CV fits: 0 of 360
Milestone 3 LSTM: checkpoint integrity VERIFIED; MAE=0.129960 kW
Milestone 3 TCN: checkpoint integrity VERIFIED; MAE=0.387378 kW
NEURAL MILESTONE 4 PREFLIGHT: PASSED — no training performed


In [18]:
# Explicit one-fit cap. This cell NEVER starts training.
M4_RUN_ONE = False                 # Change to True only when ready.
M4_MODEL = 'LSTM'                  # 'LSTM' or 'TCN'
M4_HORIZON = 15                    # 15, 30, 60
M4_FAMILY = 'F0'                   # F0 through F4
M4_FOLD = 1                       # 1, 2, 3
M4_CONFIG_INDEX = 0               # 0, 1, 2, 3
print('One-fit campaign switch:', 'ENABLED' if M4_RUN_ONE else 'DISABLED')


One-fit campaign switch: DISABLED


In [19]:
# ONLY this cell may start one full nested-CV training evaluation.
# Leave M4_RUN_ONE=False during initial notebook execution.
if not M4_RUN_ONE:
    print('NEURAL MILESTONE 4 TRAINING: DISABLED — no fit executed')
else:
    if not globals().get('m4_preflight_passed', False):
        raise RuntimeError(
            'Milestone 4 full preflight is required: execute Section C, '
            'then rerun the M4 preflight before enabling one fit.'
        )
    base = base_modelling_tables[M4_HORIZON]
    forbidden = holdout_masks[M4_HORIZON]
    assert int(forbidden.sum()) == 26496
    target_masked = base['target_P_Solar_kW'].mask(forbidden)
    assert target_masked.loc[forbidden].isna().all()
    fold = make_rolling_origin_masks(base['valid_time'])[M4_FOLD - 1]
    m4_payload = prepare_development_fold(
        X=feature_family_tables[M4_HORIZON][M4_FAMILY],
        masked_target=target_masked,
        issue_time=base['issue_time'], valid_time=base['valid_time'],
        forbidden_mask=forbidden, fold=fold,
        horizon_minutes=M4_HORIZON, family=M4_FAMILY,
    )
    validate_campaign_payload(m4_payload)
    m4_record, m4_is_new = run_one_cv(
        payload=m4_payload, root=m4_root, model_name=M4_MODEL,
        config_index=M4_CONFIG_INDEX, source_sha256=m4_sha,
        output_dir=m4_dir,
    )
    print('NEURAL MILESTONE 4 SINGLE FIT: PASSED —',
          'new fit' if m4_is_new else 'matching checkpoint reused')
    print('Best epoch:', m4_record['best_epoch'])
    print('MAE kW:', m4_record['validation_mae_kw'])
    print('RMSE kW:', m4_record['validation_rmse_kw'])
    print('Training seconds:', round(m4_record['inner_selection_seconds']+
                                   m4_record['outer_refit_seconds'], 1))
    print('Archived CV fits:', len(read_campaign_progress(m4_dir)), 'of 360')


NEURAL MILESTONE 4 TRAINING: DISABLED — no fit executed


## Milestone 4 initial completion gate — stop here

The default run should print `NEURAL MILESTONE 4 PREFLIGHT: PASSED` and
`NEURAL MILESTONE 4 TRAINING: DISABLED`. Do not start a full-grid search yet.
The audited M3 feasibility gate records must stay separate from M4 CV score
checkpoints. Candidate ranking is not permitted until **all four candidates
have all three outer-fold scores** for a model/horizon/family block.

For a later explicitly approved trial, edit the Milestone 4 switch cell,
run it, then run **only** the Milestone 4 training cell. Each invocation
trains at most one candidate/fold and resumes exact completed checkpoints.

A fresh-kernel run may print **PENDING** instead of **PASSED** if Section C has not yet materialized the development tensors. The M4 training cell refuses opt-in training until the full checkpoint preflight succeeds.


### One-click verification gate (no training)

This final cell confirms all earlier data/parity checks and read-only
checkpoint audits are available in the current kernel. It does not
train or select a neural model. **Passing this gate means only that
the notebook preflight is complete; the 360-fold CV campaign has
not been run.**


In [20]:
# Complete read-only run-all gate; no TensorFlow training or result writes.
from pathlib import Path

assert isinstance(PROJECT_ROOT, Path), 'Project root was not initialized'
assert (PROJECT_ROOT / 'src/revised_neural_campaign.py').is_file()
assert (15, 'F0', 1) in neural_fold_smokes, 'Run canonical Section C first'
assert globals().get('m4_preflight_passed', False), (
    'Milestone 4 checkpoint audit has not passed; do not enable any training'
)
assert M3_RUN_LSTM is False and M3_RUN_TCN is False, (
    'Full-size gates should remain disabled for a safe Run All'
)
assert M4_RUN_ONE is False, 'The full-CV single-fit opt-in must remain disabled'
assert all(m4_audit[name] is not None for name in ('LSTM', 'TCN'))
assert int(holdout_masks[15].sum()) == 26496
assert len(read_campaign_progress(m4_dir)) <= 360

print('NEURAL NOTEBOOK 11 RUN-ALL PREFLIGHT: PASSED')
print('Completed LSTM/TCN single-fold gate checkpoint audit: PASSED')
print('Full 360-evaluation development CV: NOT STARTED/NOT COMPLETED by this run')
print('Terminal holdout: NOT EVALUATED')
print('No new full-size neural training was requested')


NEURAL NOTEBOOK 11 RUN-ALL PREFLIGHT: PASSED
Completed LSTM/TCN single-fold gate checkpoint audit: PASSED
Full 360-evaluation development CV: NOT STARTED/NOT COMPLETED by this run
Terminal holdout: NOT EVALUATED
No new full-size neural training was requested
